# Experiment 4: Exhaustive Search and PSD Testing for 3 Classes

*In this notebook, exhaustive search tests are conducted across the three rainfall intensity classes. The workflow begins with dependency importation and data loading, followed by the feature selection stage. Finally, the process moves to the Power Spectral Density (PSD) evaluation and the execution of the exhaustive search across the classification algorithms.*

## Imports and Configurations

In [ ]:
# Experiment
EXP_NUM = 4
DATASET = 'IDSM'
CONTENT = 'dry/wet'
GRANULARITY = 5
TARGET_COL = 'category'
CLASS_MAP = {
    'no-rain': 'No Rain', 
    'light': 'Moderate', 'moderate': 'Moderate', 
    'heavy': 'Heavy', 'violent': 'Heavy'
}

import pandas as pd
import sys
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings('ignore')

current_path = Path.cwd()
PROJECT_ROOT = None

for p in [current_path, current_path.parent, current_path.parent.parent]:
    if (p / "rainfall_acoustic_classification").exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Project root added to sys.path")
else:
    print("ERROR: Unable to find the project root.")

from rainfall_acoustic_classification.visualization import VisualizationEngine
from rainfall_acoustic_classification.utils import get_standard_logger

# Feature Engineering
from rainfall_acoustic_classification.feature_engineering import (
    # Core modules
    ExperimentCreator, ExperimentConfig, 
    
    # Single Metric (PSD_mean) 
    SingleSelectorConfig, build_single_selector,

    # Metrics Selection
    VectorSelectorConfig, VectorSelector,
    
    # Vizualization
    plot_correlation_heatmap, plot_individual_boxplots, plot_feature_importance, plot_fisher_scores
)

# Modeling
from rainfall_acoustic_classification.modeling import (
    # Classification
    ClassifierFactory, ClassifierConfig,

    # Hyperparameter Optimization
    ModelOptimizer,TuningConfig,

    # Validation
    ModelEvaluator, ValidationConfig,

    #Vizualization
    plot_confusion_matrix_grid, plot_multiclass_pr_curve, plot_experiment_performance_heatmap
)

# Directories
DATA_DIR = PROJECT_ROOT / "data" / "processed" / DATASET
REPORTS_DIR = PROJECT_ROOT / "reports" / "vector_selector" / "experiment_4" / DATASET
MODELS_DIR = PROJECT_ROOT / "models" / "vector_selector" / "experiment_4" / DATASET

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## Data Loading and Mapping

In [ ]:
df_train_metrics = pd.read_csv(DATA_DIR /  f"{DATASET}_train_metrics.csv")
df_val_metrics = pd.read_csv(DATA_DIR / f"{DATASET}_val_metrics.csv")
df_test_metrics = pd.read_csv(DATA_DIR / f"{DATASET}_test_metrics.csv")

print(f"Raw Shapes -> Train: {df_train_metrics.shape}, Val: {df_val_metrics.shape}, Test: {df_test_metrics.shape}")

In [ ]:
metadata_cols = [
    'file_name', 'timestamp', 'period', 'mm_5min', 'mm_hr', 
    'category', 'recorder', 'location', 'file_path', 'extension', 
    'year', 'month', 'day', 'hour', 'minute', 'second', 'wet', 'split', 
    'should_augment', 'segment_idx', 'offset_sec', 'aug_params'
    ]

exp_config = ExperimentConfig(content=CONTENT,
                              granularity=GRANULARITY, 
                              label_col=TARGET_COL,
                              metadata_cols = metadata_cols,
                              custom_mapping=CLASS_MAP)

X_train_metrics, y_train_metrics, _ = ExperimentCreator.extract_X_y_meta(df_train_metrics, config=exp_config)
X_train, y_train = ExperimentCreator.apply_experiment_rules(X_train_metrics, y_train_metrics, config=exp_config)


X_val_metrics, y_val_metrics, _ = ExperimentCreator.extract_X_y_meta(df_val_metrics, config=exp_config)
X_val, y_val = ExperimentCreator.apply_experiment_rules(X_val_metrics, y_val_metrics, config=exp_config)


X_test_metrics, y_test_metrics, _ = ExperimentCreator.extract_X_y_meta(df_test_metrics, config=exp_config)
X_test, y_test = ExperimentCreator.apply_experiment_rules(X_test_metrics, y_test_metrics, config=exp_config)

print(f"Classes post mapping -> Train: {y_train.unique()}, Val: {y_val.unique()}, Test: {y_test.unique()}")

## Feature Engineering (Metrics Selection and PSD)

*Run this code so that the algorithm applies a filter to the features in the dataset and selects only those that pass the filter*

In [ ]:
metrics_selector_config = VectorSelectorConfig(corr_threshold=0.85, 
                                               rf_n_estimators=100, 
                                               fisher_percentile=10)
metrics_selector = VectorSelector(metrics_selector_config)

X_train_selected_metrics = metrics_selector.fit_transform(X_train, y_train)

survivors = metrics_selector.get_feature_names_out()
print(f"Original Features: {X_train.shape[1]} -> Selected: {X_train_selected_metrics.shape[1]}")
print(f"Selected ones: {survivors}")

X_val_selected_metrics = metrics_selector.transform(X_val)
X_test_selected_metrics = metrics_selector.transform(X_test)

df_feature_tracking = metrics_selector.get_metrics_report()

## Graphics

In [ ]:
display(df_feature_tracking.head(50))

In [ ]:
display(df_feature_tracking.tail(52))

In [ ]:
viz_engine = VisualizationEngine()
palette = viz_engine.get_rain_color_palette()
rain_palette = dict(list(palette.items())[:5])
exp_palette = {k: v for k, v in rain_palette.items() if k in y_train.unique()}
ordered_classes = list(exp_palette.keys())

selected_metrics = df_feature_tracking[df_feature_tracking['Survived_Pipeline'] == 1]['Feature_Name'].tolist()
X_train_fischer_metrics = X_train[selected_metrics]

In [ ]:
plot_correlation_heatmap(
    df=X_train_fischer_metrics,
    threshold=0.85,
    title=f"Selected Metrics Correlation"
)

In [ ]:
plot_individual_boxplots(
    X=X_train_fischer_metrics, 
    y=y_train, 
    color_map=rain_palette,
    class_order=ordered_classes
)

In [ ]:
plot_fisher_scores(
    df_tracking=df_feature_tracking, 
    top_n=10,
    title="Fisher Score"
)

## Feature Filtering (PSD)

*In this cell, a filter is applied to the dataset to exclusively isolate the Power Spectral Density (PSD) features. This step ensures that the subsequent feature selection process operates solely on acoustic energy metrics focused within the frequency domain.*

In [ ]:
psd_selector_config = SingleSelectorConfig(target_feature='psd_mean', 
                                           fisher_percentile=60)
psd_selector = build_single_selector(psd_selector_config)


X_train_psd = psd_selector.fit_transform(X_train, y_train)
print(f"Original Features: {X_train.shape[1]} -> Selected: {X_train_psd.shape[1]}")
print(f"Selected ones: {X_train_psd.columns}")

X_val_psd = psd_selector.transform(X_val)
X_test_psd = psd_selector.transform(X_test)

## Target Variable Encoding

*Application of the `LabelEncoder` to convert categorical text classes (e.g., 'Heavy', 'Light') into integer numerical representations, ensuring compatibility with the mathematical requirements of Machine Learning algorithms.*

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Invoke the encoder (text-to-number converter)
le = LabelEncoder()

# Converts the text strings (“Heavy”, “Light”) into integers (0, 1, 2, 3, 4)
# and preserves the original Pandas index
y_train_enc = pd.Series(le.fit_transform(y_train), index=y_train.index, name=TARGET_COL)
y_val_enc   = pd.Series(le.transform(y_val), index=y_val.index, name=TARGET_COL)
y_test_enc  = pd.Series(le.transform(y_test), index=y_test.index, name=TARGET_COL)

print("The variables “y_train_enc”, “y_val_enc” and “y_test_enc” have been recreated and saved in memory!")
print(f"Internal mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

## PSD

*In this session, tests are carried out using PSD to assess the metric’s performance in distinguishing between the five rainfall classes.*

In [ ]:
sgd_config = ClassifierConfig(model_type='sgd', random_state=42, n_jobs=-1)
sgd_base = ClassifierFactory.build(sgd_config)

sgd_grid = {
    'alpha': [0.0001, 0.001],
    'penalty': ['l2', 'elasticnet'],
    'class_weight': ['balanced']
}

tuning_config = TuningConfig(param_grid=sgd_grid, scoring_metric='f1_macro', n_jobs=-1)

sgd_champion = ModelOptimizer.optimize(
    estimator=sgd_base, 
    X_train=X_train_psd, 
    y_train=y_train, 
    X_val=X_val_psd, 
    y_val=y_val, 
    config=tuning_config
)

val_config = ValidationConfig(average_method='macro', return_report_dict=True)

metrics_sgd = ModelEvaluator.evaluate(
    model=sgd_champion, 
    X_test=X_test_psd, 
    y_test=y_test, 
    config=val_config
)

for key, value in metrics_sgd.items():
    if isinstance(value, (int, float, str)):
        print(f" -> {key}: {value}")

In [ ]:
lr_config = ClassifierConfig(model_type='lr', random_state=42)
lr_base = ClassifierFactory.build(lr_config)

lr_grid = {
    'C': [0.1, 1.0, 10.0], 
    'class_weight': ['balanced'], 
    'solver': ['lbfgs'], 
    'max_iter': [1000]
}

tuning_config_lr = TuningConfig(param_grid=lr_grid, scoring_metric='f1_macro', n_jobs=-1)

lr_champion = ModelOptimizer.optimize(
    estimator=lr_base, 
    X_train=X_train_psd, 
    y_train=y_train_enc,
    X_val=X_val_psd, 
    y_val=y_val_enc, 
    config=tuning_config_lr
)

val_config = ValidationConfig(average_method='macro', return_report_dict=True)
metrics_lr = ModelEvaluator.evaluate(
    model=lr_champion, 
    X_test=X_test_psd, 
    y_test=y_test_enc, 
    config=val_config
)

for key, value in metrics_lr.items():
    if isinstance(value, (int, float, str)):
        print(f" -> {key}: {value}")


In [ ]:
rf_config = ClassifierConfig(model_type='rf', random_state=42)
rf_base = ClassifierFactory.build(rf_config)

rf_grid = {
    'n_estimators': [100, 200], 
    'max_depth': [None, 5, 10], 
    'min_samples_leaf': [1, 2]
}
tuning_config_rf = TuningConfig(param_grid=rf_grid, scoring_metric='f1_macro', n_jobs=-1)

rf_champion = ModelOptimizer.optimize(
    estimator=rf_base, 
    X_train=X_train_psd, 
    y_train=y_train_enc, 
    X_val=X_val_psd, 
    y_val=y_val_enc, 
    config=tuning_config_rf
)

metrics_rf = ModelEvaluator.evaluate(
    model=rf_champion, 
    X_test=X_test_psd, 
    y_test=y_test_enc, 
    config=val_config
)

for key, value in metrics_rf.items():
    if isinstance(value, (int, float, str)):
        print(f" -> {key}: {value}")

In [ ]:
xgb_config = ClassifierConfig(model_type='xgb', random_state=42)
xgb_base = ClassifierFactory.build(xgb_config)

xgb_grid = {
    'n_estimators': [100, 200], 
    'max_depth': [3, 5], 
    'learning_rate': [0.01, 0.1]
}
tuning_config_xgb = TuningConfig(param_grid=xgb_grid, scoring_metric='f1_macro', n_jobs=-1)

xgb_champion = ModelOptimizer.optimize(
    estimator=xgb_base, 
    X_train=X_train_psd, 
    y_train=y_train_enc, 
    X_val=X_val_psd, 
    y_val=y_val_enc,     
    config=tuning_config_xgb
)

metrics_xgb = ModelEvaluator.evaluate(
    model=xgb_champion, 
    X_test=X_test_psd, 
    y_test=y_test_enc,   
    config=val_config
)

for key, value in metrics_xgb.items():
    if isinstance(value, (int, float, str)):
        print(f" -> {key}: {value}")

In [ ]:
from rainfall_acoustic_classification.feature_engineering import SingleSelectorConfig, build_single_selector

psd_selector_config = SingleSelectorConfig(target_feature='psd_mean', 
                                           fisher_percentile=60)
psd_selector = build_single_selector(psd_selector_config)

X_train_psd = psd_selector.fit_transform(X_train, y_train)
print(f"Original Features: {X_train.shape[1]} -> Selected: {X_train_psd.shape[1]}")

X_val_psd = psd_selector.transform(X_val)
X_test_psd = psd_selector.transform(X_test)

# Save the list of PSD features for the exhaustive search
psd_metrics = X_train_psd.columns.tolist()
print(f"Selected ones: {psd_metrics}")

In [ ]:
# ==============================================================================
# Direct Search (5 Classes) – PSD METRICS ONLY
# ==============================================================================
import itertools
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import LabelEncoder
from rainfall_acoustic_classification.modeling import ValidationConfig, ClassifierConfig, TuningConfig, ClassifierFactory, ModelOptimizer, ModelEvaluator

print("="*60)
print("Starting Direct Search (5 Classes) - PSD METRICS ONLY")
print("="*60)

val_config = ValidationConfig(average_method='macro', return_report_dict=True)

modelos_diretos = {
    'rf': TuningConfig(param_grid={'n_estimators': [100], 'max_depth': [10]}, scoring_metric='f1_macro', n_jobs=-1),
    'lr': TuningConfig(param_grid={'C': [1.0]}, scoring_metric='f1_macro', n_jobs=-1),
    'sgd': TuningConfig(param_grid={'alpha': [0.001], 'penalty': ['l2']}, scoring_metric='f1_macro', n_jobs=-1),
    'xgb': TuningConfig(param_grid={'max_depth': [5], 'learning_rate': [0.1]}, scoring_metric='f1_macro', n_jobs=-1)
}


le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)


y_tr_safe = pd.Series(y_train_enc, index=y_train.index)
y_va_safe = pd.Series(y_val_enc, index=y_val.index)

num_psd_features = len(psd_metrics)
total_combinations = (2 ** num_psd_features) - 1

for model_name, tuning_config in modelos_diretos.items():
    print(f"\nProcessing {model_name.upper()} with {num_psd_features} PSD Metrics ({total_combinations} combinations)...")
    config_model = ClassifierConfig(model_type=model_name, random_state=42)
    combinatorial_results = []
    accountant = 0
    
    for r in range(1, num_psd_features + 1):
        for subset in itertools.combinations(psd_metrics, r):
            accountant += 1
            subset_list = list(subset)
            
            try:
                # Usa os DataFrames filtrados pelo seu SingleSelector (X_train_psd)
                champion = ModelOptimizer.optimize(
                    estimator=ClassifierFactory.build(config_model),
                    X_train=X_train_psd[subset_list], y_train=y_tr_safe, 
                    X_val=X_val_psd[subset_list], y_val=y_va_safe, config=tuning_config
                )
                
                metrics = ModelEvaluator.evaluate(model=champion, X_test=X_val_psd[subset_list], y_test=y_va_safe, config=val_config)
                report = metrics.get('classification_report', {})
                y_proba = metrics.get('y_proba')
                
                row = {
                    'Algorithm': model_name.upper(),
                    'F1_Macro_Global': metrics.get('f1_macro', 0.0),
                    'F1_Dry': 0.0,       
                    'F1_Macro_Wet': 0.0, 
                    'Precision_Macro': report.get('macro avg', {}).get('precision', 0.0),
                    'Recall_Macro': report.get('macro avg', {}).get('recall', 0.0),
                    'Accuracy': report.get('accuracy', 0.0),
                    'PR_AUC_Macro': metrics.get('pr_auc_macro', 0.0),
                    'Hyperparameters': str(tuning_config.param_grid),
                    'Features_List': subset_list
                }

                f1_wet_lista = []

                for idx_array, cls_encoded in enumerate(champion.classes_):
                    class_name = le.inverse_transform([cls_encoded])[0]
                    cls_str = str(cls_encoded)
                    class_metrics = report.get(cls_str, report.get(class_name, {}))
                    f1_atual = class_metrics.get('f1-score', 0.0)
                    
                    if class_name == 'No Rain':
                        row['F1_Dry'] = f1_atual
                    else:
                        f1_wet_lista.append(f1_atual)
                    
                    row[f'{class_name}_F1'] = f1_atual
                    row[f'{class_name}_Precision'] = class_metrics.get('precision', 0.0)
                    row[f'{class_name}_Recall'] = class_metrics.get('recall', 0.0)
                    
                    if y_proba is not None and len(y_proba.shape) == 2 and y_proba.shape[1] > idx_array:
                        y_true_binary = (y_va_safe == cls_encoded).astype(int)
                        row[f'{class_name}_PR_AUC'] = average_precision_score(y_true_binary, y_proba[:, idx_array])
                    else:
                        row[f'{class_name}_PR_AUC'] = 0.0

                if len(f1_wet_lista) > 0:
                    row['F1_Macro_Wet'] = np.mean(f1_wet_lista)

                combinatorial_results.append(row)
                
            except Exception as e:
                continue 
            
            if accountant % max(1, total_combinations // 10) == 0 or accountant == total_combinations:
                best = max([res['F1_Macro_Global'] for res in combinatorial_results]) if combinatorial_results else 0.0
                print(f"   -> {model_name.upper()}: {accountant}/{total_combinations}... Best F1 Global: {best:.4f}")

    if len(combinatorial_results) > 0:
        df_comb = pd.DataFrame(combinatorial_results).sort_values(by='F1_Macro_Global', ascending=False).reset_index(drop=True)
        csv_name = f"Results_Exp.3_5Class_PSD_{model_name.upper()}_{DATASET}.csv"
        df_comb.to_csv(csv_name, index=False)
        print(f"\nSearch {model_name.upper()} completed! Saved to: {csv_name}")
        display(df_comb.head(2))
    else:
        print(f"\nNo results generated for the model {model_name.upper()}.")

## Selected Metrics

*In this cell, the algorithm will select the top 10 metrics for classification; to change the number of metrics, simply update the MAX_FEATURES variable with the number you require*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

MAX_FEATURES = 10
top_10_features = df_feature_tracking[df_feature_tracking['Survived_Pipeline'] == 1]['Feature_Name'].head(MAX_FEATURES).tolist()

## Num Metrics Test

*This section will look at how the classifiers perform as we add new metrics*

In [ ]:
sgd_config = ClassifierConfig(model_type='sgd', random_state=42)
sgd_grid = {
    'alpha': [0.0001, 0.001], 
    'penalty': ['l2', 'elasticnet'], 
    'class_weight': ['balanced']
}
tuning_config_sgd = TuningConfig(param_grid=sgd_grid, scoring_metric='f1_macro', n_jobs=-1)
val_config = ValidationConfig(average_method='macro', return_report_dict=False)

incremental_results_sgd = []

for k in range(1, MAX_FEATURES + 1):
    current_features = top_10_features[:k]
    feature_added = current_features[-1]
    
    print(f"--- Training with {k} Feature(s) [Now added: {feature_added}] ---")
    
    # Slicing the matrix using the lists of names
    X_tr_slice = X_train_selected_metrics[current_features]
    X_va_slice = X_val_selected_metrics[current_features]
    X_te_slice = X_test_selected_metrics[current_features]
    
    try:
        # A. Model Construction
        sgd_base = ClassifierFactory.build(sgd_config)
        
        # B. Tuning
        sgd_champion = ModelOptimizer.optimize(
            estimator=sgd_base, 
            X_train=X_tr_slice, 
            y_train=y_train_enc, # Using the numerical variable we created to avoid errors
            X_val=X_va_slice, 
            y_val=y_val_enc, 
            config=tuning_config_sgd
        )
        
        # C. Evaluation on Test (The final verdict for that set)
        metrics_sgd = ModelEvaluator.evaluate(
            model=sgd_champion, 
            X_test=X_te_slice, 
            y_test=y_test_enc, 
            config=val_config
        )
        
        # D. Safe Extraction of Metrics (Handling possible changes in the dictionary)
        f1  = metrics_sgd.get('f1_macro', metrics_sgd.get('f1_score', metrics_sgd.get('f1', 0.0)))
        acc = metrics_sgd.get('accuracy', metrics_sgd.get('accuracy_score', 0.0))
        
        incremental_results_sgd.append({
            'Dimension': k,
            'Last_Feature_Included': feature_added,
            'F1_Macro': f1,
            'Accuracy': acc
        })
        
    except Exception as e:
        print(f"Error in step with {k} features: {e}")

# 4. Results Table
df_sgd_incremental = pd.DataFrame(incremental_results_sgd)
print("\nIncremental Performance Table:")
display(df_sgd_incremental)

# 5. The Definitive Chart (Learning Curve by Dimension)
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=df_sgd_incremental, 
    x='Dimension', 
    y='F1_Macro', 
    marker='o', 
    color='purple', 
    linewidth=2.5,
    markersize=8
)

plt.title("Impact of the Incremental Addition of Acoustic Features (SGD)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Number of Features Used (Top Fisher Score)", fontsize=12)
plt.ylabel("F1-Score (Macro)", fontsize=12)
plt.xticks(range(1, MAX_FEATURES + 1))
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

# Save if the REPORTS_DIR directory is defined
if 'REPORTS_DIR' in locals():
    plt.savefig(REPORTS_DIR / f"{DATASET}_E{EXP_NUM}_SGD_Incremental.pdf", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# ==============================================================================
# Dimensional Unit Test: Incremental Feature Scaling in LR
# ==============================================================================
print("="*60)
print("Feature Incremental Testing - Logistic Regression (LR)")
print("="*60)

# LR Config.
lr_config = ClassifierConfig(model_type='lr', random_state=42)
lr_grid = {
    'C': [0.1, 1.0, 10.0], 
    'class_weight': ['balanced'], 
    'solver': ['lbfgs'], 
    'max_iter': [1000]
}
tuning_config_lr = TuningConfig(param_grid=lr_grid, scoring_metric='f1_macro', n_jobs=-1)

incremental_results_lr = []

for k in range(1, MAX_FEATURES + 1):
    current_features = top_10_features[:k]
    feature_added = current_features[-1]
    print(f"--- Training LR with {k} feature(s) [+ {feature_added}] ---")
    
    X_tr_slice = X_train_selected_metrics[current_features]
    X_va_slice = X_val_selected_metrics[current_features]
    X_te_slice = X_test_selected_metrics[current_features]
    
    try:
        lr_base = ClassifierFactory.build(lr_config)
        lr_champion = ModelOptimizer.optimize(
            estimator=lr_base, X_train=X_tr_slice, y_train=y_train_enc, 
            X_val=X_va_slice, y_val=y_val_enc, config=tuning_config_lr
        )
        metrics_lr = ModelEvaluator.evaluate(model=lr_champion, X_test=X_te_slice, y_test=y_test_enc, config=val_config)
        
        f1  = metrics_lr.get('f1_macro', metrics_lr.get('f1_score', metrics_lr.get('f1', 0.0)))
        acc = metrics_lr.get('accuracy', metrics_lr.get('accuracy_score', 0.0))
        
        incremental_results_lr.append({'Dimension': k, 'Last_Feature_Included': feature_added, 'F1_Macro': f1, 'Accuracy': acc})
    except Exception as e:
        print(f"Error in step with {k} features: {e}")

# Chart
df_lr_incremental = pd.DataFrame(incremental_results_lr)
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_lr_incremental, x='Dimension', y='F1_Macro', marker='s', color='blue', linewidth=2.5, markersize=8)
plt.title("Impact of the Incremental Addition of Acoustic Features (Logistic Regression)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Number of Features Used (Top Fisher Score)", fontsize=12)
plt.ylabel("F1-Score (Macro)", fontsize=12)
plt.xticks(range(1, MAX_FEATURES + 1))
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Dimensional Unit Testing: Incremental Feature Scaling in RF
# ==============================================================================
print("="*60)
print("Feature Incremental Testing - Random Forest (RF)")
print("="*60)

# RF Config.
rf_config = ClassifierConfig(model_type='rf', random_state=42)
rf_grid = {
    'n_estimators': [100, 200], 
    'max_depth': [None, 5, 10], 
    'min_samples_leaf': [1, 2]
}
tuning_config_rf = TuningConfig(param_grid=rf_grid, scoring_metric='f1_macro', n_jobs=-1)

incremental_results_rf = []

for k in range(1, MAX_FEATURES + 1):
    current_features = top_10_features[:k]
    feature_added = current_features[-1]
    print(f"--- Training RF with {k} feature(s) [+ {feature_added}] ---")
    
    X_tr_slice = X_train_selected_metrics[current_features]
    X_va_slice = X_val_selected_metrics[current_features]
    X_te_slice = X_test_selected_metrics[current_features]
    
    try:
        rf_base = ClassifierFactory.build(rf_config)
        rf_champion = ModelOptimizer.optimize(
            estimator=rf_base, X_train=X_tr_slice, y_train=y_train_enc, 
            X_val=X_va_slice, y_val=y_val_enc, config=tuning_config_rf
        )
        metrics_rf = ModelEvaluator.evaluate(model=rf_champion, X_test=X_te_slice, y_test=y_test_enc, config=val_config)
        
        f1  = metrics_rf.get('f1_macro', metrics_rf.get('f1_score', metrics_rf.get('f1', 0.0)))
        acc = metrics_rf.get('accuracy', metrics_rf.get('accuracy_score', 0.0))
        
        incremental_results_rf.append({'Dimension': k, 'Last_Feature_Included': feature_added, 'F1_Macro': f1, 'Accuracy': acc})
    except Exception as e:
        print(f" Error in step with {k} features: {e}")

# Chart
df_rf_incremental = pd.DataFrame(incremental_results_rf)
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_rf_incremental, x='Dimension', y='F1_Macro', marker='D', color='green', linewidth=2.5, markersize=8)
plt.title("Impact of the Incremental Addition of Acoustic Features (Random Forest)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Number of Features Used (Top Fisher Score)", fontsize=12)
plt.ylabel("F1-Score (Macro)", fontsize=12)
plt.xticks(range(1, MAX_FEATURES + 1))
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Feature Dimension Testing: Incremental Feature Scaling in XGBoost
# ==============================================================================
print("="*60)
print("Feature Incremental Testing - XGBoost (XGB)")
print("="*60)

#  XGBoost Config.
xgb_config = ClassifierConfig(model_type='xgb', random_state=42)
xgb_grid = {
    'n_estimators': [100, 200], 
    'max_depth': [3, 5], 
    'learning_rate': [0.01, 0.1]
}
tuning_config_xgb = TuningConfig(param_grid=xgb_grid, scoring_metric='f1_macro', n_jobs=-1)

incremental_results_xgb = []

for k in range(1, MAX_FEATURES + 1):
    current_features = top_10_features[:k]
    feature_added = current_features[-1]
    print(f"--- Training XGB with {k} feature(s) [+ {feature_added}] ---")
    
    X_tr_slice = X_train_selected_metrics[current_features]
    X_va_slice = X_val_selected_metrics[current_features]
    X_te_slice = X_test_selected_metrics[current_features]
    
    try:
        xgb_base = ClassifierFactory.build(xgb_config)
        xgb_champion = ModelOptimizer.optimize(
            estimator=xgb_base, X_train=X_tr_slice, y_train=y_train_enc, 
            X_val=X_va_slice, y_val=y_val_enc, config=tuning_config_xgb
        )
        metrics_xgb = ModelEvaluator.evaluate(model=xgb_champion, X_test=X_te_slice, y_test=y_test_enc, config=val_config)
        
        f1  = metrics_xgb.get('f1_macro', metrics_xgb.get('f1_score', metrics_xgb.get('f1', 0.0)))
        acc = metrics_xgb.get('accuracy', metrics_xgb.get('accuracy_score', 0.0))
        
        incremental_results_xgb.append({'Dimension': k, 'Last_Feature_Included': feature_added, 'F1_Macro': f1, 'Accuracy': acc})
    except Exception as e:
        print(f"Error in step with {k} features: {e}")

# Chart
df_xgb_incremental = pd.DataFrame(incremental_results_xgb)
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_xgb_incremental, x='Dimension', y='F1_Macro', marker='^', color='orange', linewidth=2.5, markersize=8)
plt.title("Impact of the Incremental Addition of Acoustic Features (XGBoost)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Number of Features Used (Top Fisher Score)", fontsize=12)
plt.ylabel("F1-Score (Macro)", fontsize=12)
plt.xticks(range(1, MAX_FEATURES + 1))
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Dimensional Unit Test: Incremental Feature Scaling (NuSVC Poly)
# ==============================================================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import NuSVC

from rainfall_acoustic_classification.modeling.validation import ModelEvaluator, ValidationConfig

print("="*60)
print("Dimensional Unit Test: Incremental Feature Scaling (NuSVC Poly)")
print("="*60)

MAX_FEATURES = 10

# Validation configuration statement
val_config = ValidationConfig(average_method='macro', return_report_dict=False)

incremental_results_master = []

# Incremental Loop
for k in range(1, MAX_FEATURES + 1):
    current_features = top_10_features[:k]
    feature_adicionada = current_features[-1]
    
    print(f"--- Training Master Model with {k} Feature(s) [+ {feature_adicionada}] ---")
    
    # Slicing
    X_tr_slice = X_train_selected_metrics[current_features]
    X_te_slice = X_test_selected_metrics[current_features]
    
    try:
        # 1. Direct Instantiation (Bypassing the Optimiser and Factory)
        clf_master = NuSVC(
            kernel='poly', 
            degree=3, 
            coef0=1.0, 
            nu=0.5, 
            gamma='scale', 
            class_weight='balanced', 
            probability=True, 
            random_state=42
        )
        
        # 2. Direct Training on the Training Set
        clf_master.fit(X_tr_slice, y_train_enc)
        
        # 3. Inviolable Evaluation on the Test Set
        metrics_master = ModelEvaluator.evaluate(
            model=clf_master, 
            X_test=X_te_slice, 
            y_test=y_test_enc, 
            config=val_config
        )
        
        # Pulling the F1 Macro safely
        f1 = metrics_master.get('f1_macro', metrics_master.get('f1_score', metrics_master.get('f1', 0.0)))
        
        incremental_results_master.append({
            'Dimension': k,
            'Last_Feature_Included': feature_adicionada,
            'F1_Macro': f1
        })
        
    except Exception as e:
        print(f"Error in step with {k} features: {e}")

# ------------------------------------------------------------------------------
# Resultados e Gráfico
# ------------------------------------------------------------------------------
df_master_incremental = pd.DataFrame(incremental_results_master)
print("\nIncremental Performance Table (Master Model):")
display(df_master_incremental)

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=df_master_incremental, 
    x='Dimension', 
    y='F1_Macro', 
    marker='*', 
    color='gold', 
    markeredgecolor='black',
    linewidth=3.0,
    markersize=14
)

plt.title("Impact of the Incremental Addition of Features on the Master Model (NuSVC Poly Degree 3)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Number of Acoustic Features Used", fontsize=12)
plt.ylabel("F1-Score (Macro)", fontsize=12)

if not df_master_incremental.empty:
    plt.xticks(df_master_incremental['Dimension'].tolist())
    
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

if 'REPORTS_DIR' in locals():
    plt.savefig(REPORTS_DIR / f"{DATASET}_E{EXP_NUM}_MasterModel_Incremental.pdf", dpi=300, bbox_inches='tight')

plt.show()

## Exhaustive Search

*In this part of the testing, the focus is on the exhaustive search for feature combinations, which will generate .csv files ranked by best classification results*

### Kernel Optimization for SVM (NuSVC)

*This cell aims to execute the targeted exhaustive search for the Support Vector Machine (NuSVC), evaluating different kernel functions (e.g., polynomial, RBF) to maximize accuracy in separating the rainfall class boundaries.*

In [ ]:
# ==============================================================================
# Sniper Tuning (Noisy Environment/UECE): Exploring RBF and Linear Topologies
# ==============================================================================
import time
from rainfall_acoustic_classification.modeling.classifiers import ClassifierFactory, ClassifierConfig
from rainfall_acoustic_classification.modeling.tuning import ModelOptimizer, TuningConfig
from rainfall_acoustic_classification.modeling.validation import ModelEvaluator, ValidationConfig

print("="*60)
print(f"Sniper Tuning - NuSVC (Retrieval of the dataset: {DATASET})")
print("="*60)

# 1. Configuring the Base Model via Factory
nusvc_config = ClassifierConfig(model_type='nusvc', random_state=42)
nusvc_base = ClassifierFactory.build(nusvc_config)

# 2. Configuring the Hyperparameter Grid for Anti-Noisiness
# We focus on RBF (Bubbles) and Linear (Dry Cut). 
# The 'nu' parameter gets a wide spectrum to handle high class overlap.
nusvc_uece_grid = {
    'kernel': ['rbf', 'linear'],
    'nu': [0.05, 0.1, 0.2, 0.3, 0.4, 0.5],
    'gamma': ['scale', 'auto'], 
    'class_weight': ['balanced']
}

tuning_config_uece = TuningConfig(
    param_grid=nusvc_uece_grid, 
    scoring_metric='f1_macro', 
    n_jobs=-1
)

val_config = ValidationConfig(average_method='macro', return_report_dict=False)

# We will use the top 10 native features that you have already extracted using VectorSelector
X_tr_uece = X_train_selected_metrics[top_10_features]
X_va_uece = X_val_selected_metrics[top_10_features]
X_te_uece = X_test_selected_metrics[top_10_features]

try:
    print("Starting the Anti-Noise Kernel scan...")
    start_time = time.time()
    
    # We’re using the optimiser because we don’t know the best settings
    champion_nusvc_uece = ModelOptimizer.optimize(
        estimator=nusvc_base, 
        X_train=X_tr_uece, 
        y_train=y_train_enc, 
        X_val=X_va_uece, 
        y_val=y_val_enc, 
        config=tuning_config_uece
    )
    
    print("\nAssessing the Tamper-proof Test suite...")
    metrics_uece = ModelEvaluator.evaluate(
        model=champion_nusvc_uece, 
        X_test=X_te_uece, 
        y_test=y_test_enc, 
        config=val_config
    )
    
    f1 = metrics_uece.get('f1_macro', metrics_uece.get('f1_score', 0.0))
    acc = metrics_uece.get('accuracy', 0.0)
    
    elapsed = time.time() - start_time
    
    print("\n" + "="*50)
    print(f" FINAL RESULT - {DATASET} (Sniper Tuning) ")
    print("="*50)
    print(f"Search time : {elapsed/60:.2f} minutes")
    print(f"F1-Macro       : {f1:.4f}  <-- A benchmark to beat!")
    print(f"Accuracy       : {acc:.4f}")
    print("="*50)
    print(" Check the logs above to see which kernel and which 'nu' came out on top!")
    
except Exception as e:
    print(f" Error in the scan: {e}")

### Sorting process using an exhaustive search

In [ ]:
# ==============================================================================
# COMPREHENSIVE SEARCH – 3 MACRO CLASSES (RF, LR, SGD, XGB)
# ==============================================================================
import itertools
import os
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score
from rainfall_acoustic_classification.modeling import ValidationConfig, ClassifierConfig, TuningConfig, ClassifierFactory, ModelOptimizer, ModelEvaluator

print("="*60)
print("STARTING A THOROUGH SEARCH - 3 CLASSES (RF, LR, SGD, XGB)")
print("="*60)

val_config = ValidationConfig(average_method='macro', return_report_dict=True)

# 1. Dynamic Encoding
le = LabelEncoder()

# ENTER PATH TO THE CSV FOLDERS 
SAVE_DIR = r"C:\Users\Afonso\Desktop\RAC-semiurban-forest-ML-2026\notebooks\IDSM\reports"

# Creates the folder automatically if it does not already exist on your computer
os.makedirs(SAVE_DIR, exist_ok=True)

y_tr_enc = pd.Series(le.fit_transform(y_train), index=y_train.index)
y_va_enc = pd.Series(le.transform(y_val), index=y_val.index)
classes_nomes = le.classes_ 

# 2. Configuration of the 4 Models
models = {
    'rf': TuningConfig(
        param_grid={'n_estimators': [50, 100], 'max_depth': [5, 10, None], 'class_weight': ['balanced']}, 
        scoring_metric='f1_macro', n_jobs=-1
    ),
    'lr': TuningConfig(
        param_grid={'C': [0.1, 1.0, 10.0], 'solver': ['lbfgs', 'liblinear'], 'class_weight': ['balanced']}, 
        scoring_metric='f1_macro', n_jobs=-1
    ),
    'sgd': TuningConfig(
        param_grid={'alpha': [0.0001, 0.001], 'penalty': ['l2', 'elasticnet'], 'loss': ['log_loss'], 'class_weight': ['balanced']}, 
        scoring_metric='f1_macro', n_jobs=-1
    ),
    'xgb': TuningConfig(
        param_grid={'max_depth': [3, 5], 'learning_rate': [0.01, 0.1], 'n_estimators': [50, 100]}, 
        scoring_metric='f1_macro', n_jobs=-1
    )
}

features_Search = top_10_features 
total_combinations = (2 ** len(features_Search)) - 1

for model_name, tuning_config in models.items():
    print(f"\nProcessing {model_name.upper()} ({total_combinations} features combinations)...")
    config_model = ClassifierConfig(model_type=model_name, random_state=42)
    combinatorial_results = []
    accountant = 0
    
    for r in range(1, len(features_Search) + 1):
        for subset in itertools.combinations(features_Search, r): 
            accountant += 1
            subset_list = list(subset)
            
            try:
                campeao = ModelOptimizer.optimize(
                    estimator=ClassifierFactory.build(config_model),
                    X_train=X_train_selected_metrics[subset_list], y_train=y_tr_enc,
                    X_val=X_val_selected_metrics[subset_list], y_val=y_va_enc, config=tuning_config
                )
                
                metrics = ModelEvaluator.evaluate(model=campeao, X_test=X_val_selected_metrics[subset_list], y_test=y_va_enc, config=val_config)
                report = metrics.get('classification_report', {})
                y_proba = metrics.get('y_proba')
                
                # Extrai os melhores hiperparâmetros testados
                best_params = str(tuning_config.param_grid)
                if hasattr(campeao, 'get_params'):
                    best_params = str({k: campeao.get_params()[k] for k in tuning_config.param_grid.keys() if k in campeao.get_params()})
                
                row = {
                    'Algorithm': model_name.upper(),
                    'F1_Macro_Global': metrics.get('f1_macro', 0.0),
                    'Precision_Macro': report.get('macro avg', {}).get('precision', 0.0),
                    'Recall_Macro': report.get('macro avg', {}).get('recall', 0.0),
                    'Accuracy': report.get('accuracy', 0.0),
                    'PR_AUC_Macro': metrics.get('pr_auc_macro', 0.0),
                    'Best_Hyperparameters': best_params,
                    'Feature_List': subset_list
                }

                # Dynamic Extraction by Class
                for idx, nome_classe in enumerate(classes_nomes):
                    class_metrics = report.get(str(idx), report.get(nome_classe, {}))
                    row[f'{nome_classe}_F1'] = class_metrics.get('f1-score', 0.0)
                    row[f'{nome_classe}_Precision'] = class_metrics.get('precision', 0.0)
                    row[f'{nome_classe}_Recall'] = class_metrics.get('recall', 0.0)
                    
                    if y_proba is not None and len(y_proba.shape) == 2 and y_proba.shape[1] > idx:
                        y_true_binary = (y_va_enc == idx).astype(int)
                        row[f'{nome_classe}_PR_AUC'] = average_precision_score(y_true_binary, y_proba[:, idx])
                    else:
                        row[f'{nome_classe}_PR_AUC'] = 0.0

                combinatorial_results.append(row)
                
            except Exception as e: 
                continue
                
        # Print progress
        if accountant % max(1, total_combinations // 10) == 0 or accountant == total_combinations:
            best = max([res['F1_Macro_Global'] for res in combinatorial_results]) if combinatorial_results else 0.0
            print(f"   -> {model_name.upper()}: {accountant}/{total_combinations}... Best F1: {best:.4f}")

    # Save and view the CSV file for each template
    if len(combinatorial_results) > 0:
        df_res = pd.DataFrame(combinatorial_results).sort_values(by='F1_Macro_Global', ascending=False).reset_index(drop=True)
        csv_name = f"Results_Exp.4_3Class_{model_name.upper()}_{DATASET}.csv"
        df_res.to_csv(os.path.join(SAVE_DIR, csv_name), index=False)
        print(f"\nSearch{model_name.upper()} completed! Saved to: {csv_name}")
        display(df_res.head(2))
    else:
        print(f"\nNo results generated for {model_name.upper()}.")

In [ ]:
# ==============================================================================
# Direct Exhaustive Search (3 Classes) – NuSVC Bare-Metal (F1 Dry/Wet Division)
# ==============================================================================
import itertools
import time
import os
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.svm import NuSVC
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import LabelEncoder 
from rainfall_acoustic_classification.modeling import ValidationConfig, ModelEvaluator

print("="*60)
print("Starting Direct Exhaustive Search (3 Classes) - NuSVC Poly Degree 3")
print("="*60)

# 1. Safe Dynamic Encoding for the 3 classes
# Ensures that le.inverse_transform below knows exactly the names of the 3 classes
le = LabelEncoder()

# ENTER PATH TO THE CSV FOLDERS 
SAVE_DIR = r"C:\Users\Afonso\Desktop\RAC-semiurban-forest-ML-2026\notebooks\IDSM\reports"

# Creates the folder automatically if it does not already exist on your computer
os.makedirs(SAVE_DIR, exist_ok=True)

y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)

# Define the search variables
features_Search = top_10_features 
MIN_FEATURES = 6
MAX_FEATURES = len(features_Search)

val_config = ValidationConfig(average_method='macro', return_report_dict=True)

y_tr_safe = pd.Series(y_train_enc, index=y_train.index) 
y_va_safe = pd.Series(y_val_enc, index=y_val.index) 

combinatorial_results = []
accountant = 0
total_combinations_pruned = sum([len(list(itertools.combinations(features_Search, r))) for r in range(MIN_FEATURES, MAX_FEATURES + 1)])

for r in range(MIN_FEATURES, MAX_FEATURES + 1):
    for subset in itertools.combinations(features_Search, r):
        accountant += 1
        subset_list = list(subset)
        
        try:
            # Untouched bare-metal configuration
            clf_master = NuSVC(
                kernel='poly', degree=3, coef0=1.0, nu=0.5, 
                gamma='scale', class_weight='balanced', 
                probability=True, random_state=42
            )
            clf_master.fit(X_train_selected_metrics[subset_list], y_tr_safe)
            
            metrics = ModelEvaluator.evaluate(model=clf_master, X_test=X_val_selected_metrics[subset_list], y_test=y_va_safe, config=val_config)
            report = metrics.get('classification_report', {})
            y_proba = metrics.get('y_proba')
            
            row = {
                'Algorithm': 'NUSVC',
                'F1_Macro_Global': metrics.get('f1_macro', 0.0),
                'F1_Dry': 0.0,
                'F1_Macro_Wet': 0.0,
                'Precision_Macro': report.get('macro avg', {}).get('precision', 0.0),
                'Recall_Macro': report.get('macro avg', {}).get('recall', 0.0),
                'Accuracy': report.get('accuracy', 0.0),
                'PR_AUC_Macro': metrics.get('pr_auc_macro', 0.0),
                'Best_Hyperparameters': "kernel='poly', degree=3, nu=0.5",
                'Feature_List': subset_list
            }

            f1_wet_lista = []

            # Extraction with F1 Dry/Wet Separation
            for idx_array, cls_encoded in enumerate(clf_master.classes_):
                class_name = le.inverse_transform([cls_encoded])[0]
                cls_str = str(cls_encoded)
                class_metrics = report.get(cls_str, report.get(class_name, {}))
                
                f1_current = class_metrics.get('f1-score', 0.0)
                
                if class_name == 'No Rain':
                    row['F1_Dry'] = f1_current
                else:
                    f1_wet_lista.append(f1_current)
                
                row[f'{class_name}_F1'] = f1_current
                row[f'{class_name}_Precision'] = class_metrics.get('precision', 0.0)
                row[f'{class_name}_Recall'] = class_metrics.get('recall', 0.0)
                
                if y_proba is not None and len(y_proba.shape) == 2 and y_proba.shape[1] > idx_array:
                    y_true_binary = (y_va_safe == cls_encoded).astype(int)
                    row[f'{class_name}_PR_AUC'] = average_precision_score(y_true_binary, y_proba[:, idx_array])
                else:
                    row[f'{class_name}_PR_AUC'] = 0.0

            if len(f1_wet_lista) > 0:
                row['F1_Macro_Wet'] = np.mean(f1_wet_lista)

            combinatorial_results.append(row)
            
        except Exception as e:
            continue 
        
        if accountant % max(1, total_combinations_pruned // 10) == 0 or accountant == total_combinations_pruned:
            best = max([res['F1_Macro_Global'] for res in combinatorial_results]) if combinatorial_results else 0.0
            print(f"   -> Progress: {accountant}/{total_combinations_pruned}... Best F1 Global: {best:.4f}")

if len(combinatorial_results) > 0:
    df_comb = pd.DataFrame(combinatorial_results).sort_values(by='F1_Macro_Global', ascending=False).reset_index(drop=True)
    #Save results
    csv_name = f"Results_Exp.4_3Class_NuSVC_{DATASET}.csv"
    df_comb.to_csv(os.path.join(SAVE_DIR, csv_name), index=False)
    print(f"\nNuSVC search (3 class) completed! Saved to: {csv_name}")
    display(df_comb.head(2))

### UNIVERSAL CLASSIFICATION RESULTS PLOT

*This cell will prompt you to enter the name of the CSV file you wish to analyse; it will then generate the classifications and plot a standard and normalised confusion matrix, as well as the PR AUC graph.*

In [ ]:
# ==============================================================================
# UNIVERSAL PLOTTER: READS CSV, GENERATES CM + PR-AUC AND EXPORTS TO PDF
# ==============================================================================
import os 
import pandas as pd
import ast
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, precision_recall_curve, auc
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from xgboost import XGBClassifier
from sklearn.svm import NuSVC

# Import the plot function using the colours Navy and Darkred
from rainfall_acoustic_classification.modeling.plots import plot_confusion_matrix_grid

# -------------------------------------------------------------------------
# 1. MAIN SETTINGS
# -------------------------------------------------------------------------
# Enter the name of the 3-class CSV file
CSV_PATH = f"Results_Exp.4_3Class_RF_{DATASET}.csv" 

# Define where the data comes from 
X_TRAIN_DATA = X_train_selected_metrics
X_VAL_DATA = X_val_selected_metrics     

# ENTER PATH TO THE PDF FOLDERS 
SAVE_DIR = r"C:\Users\Afonso\Desktop\RAC-semiurban-forest-ML-2026\notebooks\IDSM\reports\figures"

# Creates the folder automatically if it does not already exist on your computer
os.makedirs(SAVE_DIR, exist_ok=True)

# -------------------------------------------------------------------------
# 2. CSV READING
# -------------------------------------------------------------------------
print(f"Reading results from: {CSV_PATH}")
df_res = pd.read_csv(CSV_PATH)

best_row = df_res.iloc[0]
something = str(best_row['Algorithm']).upper()
features = ast.literal_eval(best_row['Features_List'])
params_str = best_row.get('Best_Hyperparameters', best_row.get('Hyperparameters', ''))

print(f" Best Model: {something}")
print(f" Selected Features ({len(features)}): {features}")
print(f" Best Hyperparameters: {params_str}")

def parse_params(p_str):
    if pd.isna(p_str) or p_str == '': return {}
    d = {}
    clean_str = str(p_str).replace('{', '').replace('}', '').replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    for item in clean_str.split(','):
        if ':' in item:
            k, v = item.split(':')
            k, v = k.strip(), v.strip()
            try: d[k] = float(v) if '.' in v else int(v)
            except ValueError: d[k] = None if v == 'None' else v
    return d

p_dict = parse_params(params_str)

# -------------------------------------------------------------------------
# 3. MODEL RECONSTRUCTION
# -------------------------------------------------------------------------
def get_reconstructed_model(algo_name, p):
    if algo_name == 'LR':
        return LogisticRegression(C=p.get('C', 1.0), solver=p.get('solver', 'lbfgs'), max_iter=1000, random_state=42)
    elif algo_name == 'SGD':
        return SGDClassifier(alpha=p.get('alpha', 0.0001), penalty=p.get('penalty', 'l2'), loss='log_loss', random_state=42)
    elif algo_name == 'RF':
        return RandomForestClassifier(n_estimators=p.get('n_estimators', 100), max_depth=p.get('max_depth', 10), random_state=42)
    elif algo_name == 'XGB':
        return XGBClassifier(max_depth=p.get('max_depth', 5), learning_rate=p.get('learning_rate', 0.1), random_state=42)
    elif algo_name == 'NUSVC':
        return NuSVC(nu=p.get('nu', 0.5), kernel=p.get('kernel', 'poly'), probability=True, random_state=42)
    return LogisticRegression(random_state=42)

model = get_reconstructed_model(something, p_dict)

# -------------------------------------------------------------------------
# 4. FAST TRAINING AND INFERENCE
# -------------------------------------------------------------------------
print("Training the champion to generate the plots...")

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)

model.fit(X_TRAIN_DATA[features], y_train_enc)
y_score = model.predict_proba(X_VAL_DATA[features])
y_pred = model.predict(X_VAL_DATA[features])

# -------------------------------------------------------------------------
# 5. PLOT 1
# -------------------------------------------------------------------------
logical_order = ['No Rain', 'Moderate', 'Heavy']
indices_reordem = [list(le.classes_).index(cls) for cls in logical_order]

cm_raw_base = confusion_matrix(y_val_enc, y_pred)
cm_norm_base = confusion_matrix(y_val_enc, y_pred, normalize='true') * 100

cm_raw = cm_raw_base[indices_reordem, :][:, indices_reordem]
cm_norm = cm_norm_base[indices_reordem, :][:, indices_reordem]

cm_pdf_path = os.path.join(SAVE_DIR, f"CM_{something}_3Class_{DATASET}.pdf")

plot_confusion_matrix_grid(
    cm_raw=cm_raw,
    cm_norm=cm_norm,
    classes=logical_order,
    title=f"Best {something} - Selected Features",
    save_path=cm_pdf_path 
)
print(f"Confusion matrices saved in: {cm_pdf_path}")

# -------------------------------------------------------------------------
# 6. PLOT 2
# -------------------------------------------------------------------------
plt.figure(figsize=(7, 6))

colors = ['navy', 'turquoise', 'darkred']

for i, color in zip(range(len(logical_order)), colors):
    idx_original = list(le.classes_).index(logical_order[i])
    
    precision, recall, _ = precision_recall_curve(y_val_enc == idx_original, y_score[:, idx_original])
    pr_auc = auc(recall, precision)
    
    plt.plot(recall, precision, color=color, lw=2.5,
             label=f'{logical_order[i]} (AUC = {pr_auc:.2f})')

plt.xlabel('Recall', fontsize=11)
plt.ylabel('Precision', fontsize=11)
plt.title(f'Precision-Recall Curve: {something} Champion', fontsize=12, pad=15)
plt.legend(loc="lower left", fontsize=11, frameon=True, shadow=True)
plt.grid(alpha=0.3, linestyle='--')
plt.xlim([-0.05, 1.05])
plt.ylim([-0.05, 1.05])
plt.tight_layout()


prauc_pdf_path = os.path.join(SAVE_DIR, f"PRAUC_{something}_3Class_{DATASET}.pdf")
plt.savefig(prauc_pdf_path, format='pdf', dpi=300, bbox_inches='tight')
print(f"Precision-Recall Curve saved in: {prauc_pdf_path}")

plt.show()